## Config — Bronze → Silver Standardization Framework

Central configuration notebook

## Imports

Libraries below shared across all notebooks

In [0]:
import re
import uuid
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

## Run Parameters

Exposed as Databricks widgets so a scheduled job can override them without
editing the notebook.

| Widget | Purpose | Default |
|---|---|---|
| `metadata_table` | Fully-qualified control table holding column mappings | `control.metadata.standardization_mapping` |
| `debug` | Enables verbose `log()` output | `true` |
| `write_mode` | Delta write mode used by `standardize_table` | `overwrite` |


In [0]:
dbutils.widgets.text("metadata_table", "cittapoc.ref.standardization_mapping")
dbutils.widgets.dropdown("debug", "true", ["true", "false"])
dbutils.widgets.dropdown("write_mode", "overwrite", ["overwrite", "append"])
dbutils.widgets.text("null_threshold_pct", "0")

## Constants

These are the actual constants referenced throughout the framework
(`METADATA_TABLE`, `DEBUG`, `WRITE_MODE`, `NULL_THRESHOLD_PCT`)

In [0]:
METADATA_TABLE = dbutils.widgets.get("metadata_table")
DEBUG = dbutils.widgets.get("debug").lower() == "true"
WRITE_MODE = dbutils.widgets.get("write_mode")

print(f"METADATA_TABLE      = {METADATA_TABLE}")
print(f"DEBUG                = {DEBUG}")
print(f"WRITE_MODE           = {WRITE_MODE}")


## Expected `METADATA_TABLE` Schema

The standardization engine expects the control table above to contain atleast one row per source column, per target table:

**Required columns:**
- `src_catalog`, `src_schema`, `src_table` — Bronze source identifiers
- `tgt_catalog`, `tgt_schema`, `tgt_table` — Silver target identifiers
- `src_column`, `tgt_column` — source → target column name mapping
- `tgt_dtype` — target Spark SQL type (e.g. `"string"`, `"decimal(18,2)"`)
- `default_value` — value used when `src_column` is missing from the source
- `ordinal_position` — controls output column order
- `active` — boolean; only `True` rows are used

**Optional columns** (safe to omit — the engine falls back to sensible
defaults if these aren't present):
- `is_primary_key` (bool) — enables keyed deduplication
- `is_mandatory` (bool) — enables null-check in `data_quality_check`
- `partition_column` (bool) — enables `partitionBy` on Silver write

## Validation

Fails fast if the metadata table doesn't exist, rather than letting every
downstream notebook hit a confusing "table not found" error mid-pipeline.

In [0]:
if not spark.catalog.tableExists(METADATA_TABLE):
    raise Exception(
        f"Config error: METADATA_TABLE '{METADATA_TABLE}' does not exist. "
        "Check the 'metadata_table' widget or create the control table first."
    )